In [2]:
from _provenance import Sello
sello = Sello('06_projection_tecniques.ipynb')

In [ ]:
import os
import re
from datetime import datetime
from pathlib import Path

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.decomposition import KernelPCA
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import umap

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ---------------- Guardado de figuras ----------------
# Las figuras se guardan como PDF en SAVE_DIR y NO se muestran en el notebook.
# SAVE_DIR = BASE_SAVE_DIR/<nombre del CSV cargado> (se define al cargar los datos).
# GEN_DATE se imprime en la esquina inferior de cada figura.
BASE_SAVE_DIR = '../figures/proyeccion_tecniques'
GEN_DATE = datetime.now().strftime('%Y-%m-%d %H:%M')
METADATA=['origin_db', 'subject_id', 'stimulus', 'window']

In [18]:
ruta='../data/prueba/df_flat_wnd_max.csv'

In [ ]:
df_norm = pd.read_csv(ruta)
FEATURES=df_norm.columns.difference(METADATA)
title="max ventanas aplanadas"

# Una subcarpeta por fuente de datos para no mezclar resultados
DATASET_TAG = Path(ruta).stem
SAVE_DIR = f'{BASE_SAVE_DIR}/{DATASET_TAG}'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Dataset: {ruta} | Figuras en: {SAVE_DIR} | Generado: {GEN_DATE}')

In [7]:
df_norm

,origin_db,subject,stimulus,VAR_STD_w0,VAR_STD_w1,VAR_STD_w2,VAR_STD_w3,VAR_STD_w4,VAR_STD_w5,VAR_STD_w6,...,Max_w15,Max_w16,Max_w17,Max_w18,Max_w19,Max_w20,Max_w21,Max_w22,Max_w23,Max_w24
0,NinaProDB1,1,22,-3.671784,-3.556763,-3.354895,-2.029582,-0.753676,0.802121,2.111312,...,-0.862417,-0.785630,-0.726330,-0.736622,-0.843709,-1.088398,-1.432645,-1.894877,-2.497103,-2.860361
1,NinaProDB1,1,25,-3.675160,-3.259773,-2.717560,-2.390495,-2.273891,-1.078430,1.575245,...,0.708969,-0.171824,-1.124766,-2.049268,-2.920089,-3.710577,-4.418129,-4.907769,-5.151217,-5.204715
2,NinaProDB1,1,27,-2.551344,-2.490121,-1.642568,0.293491,0.849404,0.994104,0.455105,...,1.241130,0.919021,0.562300,0.033209,-0.683990,-1.459082,-2.175571,-2.837918,-3.507260,-3.877601
3,NinaProDB1,1,30,-3.011636,-2.827883,-1.577062,-0.904992,2.281744,3.449172,4.245980,...,1.976290,2.340103,2.572147,2.537577,2.200311,1.514228,0.455142,-0.828550,-2.105218,-2.768913
4,NinaProDB1,1,31,-3.011337,-2.880280,-2.275380,-1.833787,-0.763108,-0.078727,0.444615,...,-2.240194,-2.214682,-2.192962,-2.172365,-2.201858,-2.306687,-2.579669,-3.038975,-3.634092,-3.967386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
604,NinaProDB5,10,27,-2.875686,-1.625477,-0.306390,-0.012053,0.458778,1.290379,1.936566,...,2.240916,1.954000,1.654804,1.260167,0.740508,0.055727,-0.820838,-1.842592,-2.780186,-3.258074
605,NinaProDB5,10,30,-3.676114,-2.972145,-1.270987,-0.098635,0.973656,1.799723,2.329476,...,2.575748,2.422485,2.154769,1.786683,1.276936,0.570328,-0.375835,-1.533743,-2.602045,-3.167537
606,NinaProDB5,10,31,-4.157620,-3.714170,-2.767890,-1.767855,-1.266723,-0.991908,-0.923248,...,-0.959415,-0.942126,-0.831810,-0.736218,-0.769663,-0.959722,-1.252965,-1.664764,-2.168087,-2.472277
607,NinaProDB5,10,37,-1.494897,-0.149810,1.145083,1.975923,2.168638,2.350332,2.246928,...,2.045080,1.950473,1.777266,1.529479,1.188130,0.726958,0.159549,-0.498001,-1.213237,-1.616874


In [8]:
def _save_fig(fig, name):
    """Guarda la figura en PDF dentro de SAVE_DIR y libera la memoria (sin plt.show)."""
    os.makedirs(SAVE_DIR, exist_ok=True)
    fig.savefig(f'{SAVE_DIR}/{name}.pdf', dpi=300, bbox_inches='tight')
    plt.close(fig)


def _fig_name(title, hue_col):
    """Slug de nombre de archivo a partir del titulo y la variable de color.

    'UMAP - max ventanas aplanadas' + 'stimulus' -> 'umap_max_ventanas_aplanadas_stimulus'
    """
    return re.sub(r'[^a-z0-9]+', '_', f'{title}_{hue_col}'.lower()).strip('_')


def plot_pca_separability(df, cols, title, hue_col, palette='tab10'):
    """
    Aplica PCA a un conjunto de columnas y visualiza la separabilidad 
    de las clases en un espacio 2D.
    """
    # 1. Preparar copia y limpiar NaNs
    df_plot = df.copy()
    X = df_plot[cols].values
    X = np.nan_to_num(X, nan=0.0)

    # 2. Aplicar PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    # Añadir coordenadas al dataframe temporal
    df_plot['pca_1'] = X_pca[:, 0]
    df_plot['pca_2'] = X_pca[:, 1]

    # 3. Visualización
    plt.figure(figsize=(12, 8))
    sns.scatterplot(
        data=df_plot, 
        x='pca_1', 
        y='pca_2', 
        hue=hue_col, 
        palette=palette, 
        s=80, 
        alpha=0.8, 
        edgecolor='white', 
        linewidth=0.5
    )

    plt.title(f'{title}', 
              fontweight='bold', fontsize=15)
    plt.xlabel(f'PCA Dim 1 ({pca.explained_variance_ratio_[0]:.2%} var)')
    plt.ylabel(f'PCA Dim 2 ({pca.explained_variance_ratio_[1]:.2%} var)')

    plt.legend(title=hue_col, bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    # Pie: fecha de generacion (esquina inferior derecha)
    fig = plt.gcf()
    fig.text(0.99, 0.01, f'Generado: {GEN_DATE}', ha='right', va='bottom',
             fontsize=7, color='#666666', transform=fig.transFigure)
    _save_fig(fig, _fig_name(title, hue_col))

In [9]:
def plot_tsne_separability(df, cols, title, hue_col, palette='tab10'):
    """
    Aplica t-SNE a un conjunto de columnas y visualiza la separabilidad 
    de las clases en un espacio 2D.
    """
    # 1. Preparar copia y limpiar NaNs
    df_plot = df.copy()
    X = df_plot[cols].values
    X = np.nan_to_num(X, nan=0.0)

    # 2. Aplicar t-SNE
    tsne = TSNE(
        n_components=2,
        method='exact',
        perplexity=min(30, X.shape[0]-1), 
        learning_rate='auto', 
        init='pca', 
        random_state=42
    )
    X_tsne = tsne.fit_transform(X)

    # Añadir coordenadas al dataframe temporal
    df_plot['tsne_1'] = X_tsne[:, 0]
    df_plot['tsne_2'] = X_tsne[:, 1]

    # 3. Visualización
    plt.figure(figsize=(12, 8))
    sns.scatterplot(
        data=df_plot, 
        x='tsne_1', 
        y='tsne_2', 
        hue=hue_col, 
        palette=palette, 
        s=80, 
        alpha=0.8, 
        edgecolor='white', 
        linewidth=0.5
    )

    plt.title(f'{title}', 
              fontweight='bold', fontsize=15)
    plt.xlabel('t-SNE Dim 1')
    plt.ylabel('t-SNE Dim 2')

    plt.legend(title=hue_col, bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    # Pie: fecha de generacion (esquina inferior derecha)
    fig = plt.gcf()
    fig.text(0.99, 0.01, f'Generado: {GEN_DATE}', ha='right', va='bottom',
             fontsize=7, color='#666666', transform=fig.transFigure)
    _save_fig(fig, _fig_name(title, hue_col))

In [10]:
def plot_umap_separability(df, cols, title, hue_col, palette='tab10'):
    """
    Aplica UMAP a un conjunto de columnas y visualiza la separabilidad 
    de las clases en un espacio 2D.
    """
    # 1. Preparar copia y limpiar NaNs
    df_plot = df.copy()
    X = df_plot[cols].values
    X = np.nan_to_num(X, nan=0.0)

    # 2. Aplicar UMAP
    # n_neighbors: balance entre estructura local vs global
    # min_dist: control del agrupamiento de puntos
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=min(30, X.shape[0] - 1),
        min_dist=0.1,
        random_state=42
    )
    X_umap = reducer.fit_transform(X)

    # Añadir coordenadas al dataframe temporal
    df_plot['umap_1'] = X_umap[:, 0]
    df_plot['umap_2'] = X_umap[:, 1]

    # 3. Visualización
    plt.figure(figsize=(12, 8))
    sns.scatterplot(
        data=df_plot, 
        x='umap_1', 
        y='umap_2', 
        hue=hue_col, 
        palette=palette, 
        s=80, 
        alpha=0.8, 
        edgecolor='white', 
        linewidth=0.5
    )

    plt.title(f'{title}', 
              fontweight='bold', fontsize=15)
    plt.xlabel('UMAP Dim 1')
    plt.ylabel('UMAP Dim 2')

    plt.legend(title=hue_col, bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    # Pie: fecha de generacion (esquina inferior derecha)
    fig = plt.gcf()
    fig.text(0.99, 0.01, f'Generado: {GEN_DATE}', ha='right', va='bottom',
             fontsize=7, color='#666666', transform=fig.transFigure)
    _save_fig(fig, _fig_name(title, hue_col))

In [11]:
def plot_kpca_separability(df, cols, title,hue_col,kernel='rbf', palette='tab10'):
    """
    Aplica Kernel PCA a un conjunto de columnas y visualiza la separabilidad 
    de las clases en un espacio 2D.
    """
    # 1. Preparar copia y limpiar NaNs
    df_plot = df.copy()
    X = df_plot[cols].values
    X = np.nan_to_num(X, nan=0.0)

    # 2. Aplicar Kernel PCA
    # El kernel 'rbf' es el más común para capturar relaciones no lineales
    kpca = KernelPCA(n_components=2, kernel=kernel, fit_inverse_transform=True, random_state=42)
    X_kpca = kpca.fit_transform(X)

    # Añadir coordenadas al dataframe temporal
    df_plot['kpca_1'] = X_kpca[:, 0]
    df_plot['kpca_2'] = X_kpca[:, 1]

    # 3. Visualización
    plt.figure(figsize=(12, 8))
    sns.scatterplot(
        data=df_plot, 
        x='kpca_1', 
        y='kpca_2', 
        hue=hue_col, 
        palette=palette, 
        s=80, 
        alpha=0.8, 
        edgecolor='white', 
        linewidth=0.5
    )

    plt.title(f'{title}', 
              fontweight='bold', fontsize=15)
    plt.xlabel('KPCA Dim 1')
    plt.ylabel('KPCA Dim 2')

    plt.legend(title=hue_col, bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    # Pie: fecha de generacion (esquina inferior derecha)
    fig = plt.gcf()
    fig.text(0.99, 0.01, f'Generado: {GEN_DATE}', ha='right', va='bottom',
             fontsize=7, color='#666666', transform=fig.transFigure)
    _save_fig(fig, _fig_name(title, hue_col))

In [ ]:
plot_umap_separability(df_norm, FEATURES, f"UMAP - {title}",'origin_db')
plot_tsne_separability(df_norm, FEATURES, f"t-SNE - {title}",'origin_db')
plot_kpca_separability(df_norm, FEATURES, f"KPCA - {title}",'origin_db')

/home/Felixphz/project/sEMG-ArmBand/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
plot_umap_separability(df_norm, FEATURES, f"UMAP - {title}",'stimulus')
plot_tsne_separability(df_norm, FEATURES, f"t-SNE - {title}",'stimulus')
plot_kpca_separability(df_norm, FEATURES, f"KPCA - {title}",'stimulus')  

/home/Felixphz/project/sEMG-ArmBand/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [16]:
sello.mostrar()

### Sello de procedencia

| Campo | Valor |
|---|---|
| Notebook | `06_projection_tecniques.ipynb` |
| Commit | `f5d3083` |
| Rama | `main` |
| Estado | sucio (21 archivos sin commitear) |
| Ejecutado | 2026-10-13T18:37:59 |

**Archivos generados en esta ejecución:**

- `figures/proyeccion_tecniques/kpca_max_ventanas_aplanadas_origin_db.pdf`
- `figures/proyeccion_tecniques/kpca_max_ventanas_aplanadas_stimulus.pdf`
- `figures/proyeccion_tecniques/t_sne_max_ventanas_aplanadas_origin_db.pdf`
- `figures/proyeccion_tecniques/t_sne_max_ventanas_aplanadas_stimulus.pdf`
- `figures/proyeccion_tecniques/umap_max_ventanas_aplanadas_origin_db.pdf`
- `figures/proyeccion_tecniques/umap_max_ventanas_aplanadas_stimulus.pdf`

> Guardado en `PROVENANCE.md` (raíz del repo).